# 0. Preparation

In [ ]:
# import sys
# !{sys.executable} -m pip install -r requirements.txt

# 1. Import relevant packages and set up basic variables

In [ ]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from experiment_config import DATA_PATHS, RESULTS_DIR
from methods.unet import UNetMethod
from methods.attention_unet import AttentionUNetMethod
from methods.dlv3plus import DLv3Plus
from experiment_runner import build_comparison_table
from utils.visualization import save_prediction_samples

PROJECT_ROOT = Path('.').resolve()
RESULTS_ROOT = PROJECT_ROOT / RESULTS_DIR
SEED = 34

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

sns.set_theme(style='whitegrid')
print('Project root:', PROJECT_ROOT)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# 2. Define the experiment settings

In [ ]:
MAIN_MODEL_CONFIGS = [
    {
        'display_name': 'U-Net',
        'method_class': UNetMethod,
        'kwargs': {'pretrained': True, 'use_augmentation': True},
        'result_dir': RESULTS_ROOT / 'unet' / 'main_experiment' / 'unet_main'
    },
    {
        'display_name': 'Attention U-Net',
        'method_class': AttentionUNetMethod,
        'kwargs': {'use_augmentation': True},
        'result_dir': RESULTS_ROOT / 'attention_unet' / 'main_experiment' / 'attention_unet_main'
    },
    {
        'display_name': 'DeepLabV3+',
        'method_class': DLv3Plus,
        'kwargs': {'pretrained': True, 'use_augmentation': True},
        'result_dir': RESULTS_ROOT / 'deeplabv3plus' / 'main_experiment' / 'deeplabv3plus_main'
    }
]

ABLATION_CONFIGS = {
    'U-Net: Pretraining': [
        {
            'display_name': 'U-Net (Pretrained)',
            'method_class': UNetMethod,
            'kwargs': {'pretrained': True, 'use_augmentation': True},
            'result_dir': RESULTS_ROOT / 'unet' / 'ablation_pretraining' / 'unet_pretrained'
        },
        {
            'display_name': 'U-Net (Scratch)',
            'method_class': UNetMethod,
            'kwargs': {'pretrained': False, 'use_augmentation': True},
            'result_dir': RESULTS_ROOT / 'unet' / 'ablation_pretraining' / 'unet_scratch'
        }
    ],
    'U-Net: Augmentation': [
        {
            'display_name': 'U-Net (With Augmentation)',
            'method_class': UNetMethod,
            'kwargs': {'pretrained': True, 'use_augmentation': True},
            'result_dir': RESULTS_ROOT / 'unet' / 'ablation_augmentation' / 'unet_with_aug'
        },
        {
            'display_name': 'U-Net (Without Augmentation)',
            'method_class': UNetMethod,
            'kwargs': {'pretrained': True, 'use_augmentation': False},
            'result_dir': RESULTS_ROOT / 'unet' / 'ablation_augmentation' / 'unet_without_aug'
        }
    ],
    'Attention U-Net: Augmentation': [
        {
            'display_name': 'Attention U-Net (With Augmentation)',
            'method_class': AttentionUNetMethod,
            'kwargs': {'use_augmentation': True},
            'result_dir': RESULTS_ROOT / 'attention_unet' / 'ablation_augmentation' / 'attention_unet_with_aug'
        },
        {
            'display_name': 'Attention U-Net (Without Augmentation)',
            'method_class': AttentionUNetMethod,
            'kwargs': {'use_augmentation': False},
            'result_dir': RESULTS_ROOT / 'attention_unet' / 'ablation_augmentation' / 'attention_unet_without_aug'
        }
    ],
    'Attention Module Comparison': [
        {
            'display_name': 'U-Net (Baseline)',
            'method_class': UNetMethod,
            'kwargs': {'pretrained': True, 'use_augmentation': True},
            'result_dir': RESULTS_ROOT / 'attention_unet' / 'ablation_attention_module' / 'unet_baseline'
        },
        {
            'display_name': 'Attention U-Net',
            'method_class': AttentionUNetMethod,
            'kwargs': {'use_augmentation': True},
            'result_dir': RESULTS_ROOT / 'attention_unet' / 'ablation_attention_module' / 'attention_unet_attention_module'
        }
    ],
    'DeepLabV3+: Pretraining': [
        {
            'display_name': 'DeepLabV3+ (Pretrained)',
            'method_class': DLv3Plus,
            'kwargs': {'pretrained': True, 'use_augmentation': True},
            'result_dir': RESULTS_ROOT / 'deeplabv3plus' / 'ablation_pretraining' / 'deeplabv3plus_pretrained'
        },
        {
            'display_name': 'DeepLabV3+ (Scratch)',
            'method_class': DLv3Plus,
            'kwargs': {'pretrained': False, 'use_augmentation': True},
            'result_dir': RESULTS_ROOT / 'deeplabv3plus' / 'ablation_pretraining' / 'deeplabv3plus_scratch'
        }
    ]
}

print('Main experiments:', len(MAIN_MODEL_CONFIGS))
print('Ablation groups:', len(ABLATION_CONFIGS))

# 3. Define helper functions: run experiments, save metrics, and visualize results

In [ ]:
def ensure_dir(path):
    path.mkdir(parents=True, exist_ok=True)

def save_json(path, data):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2)

def run_single_model_experiment(config):
    ensure_dir(config['result_dir'])
    method = config['method_class'](**config['kwargs'])

    train_metrics, val_metrics = method.train(
        train_path=DATA_PATHS['train'],
        val_path=DATA_PATHS['validation']
    )
    test_metrics, _, _ = method.test(test_path=DATA_PATHS['test'])

    method.save_history_csv(config['result_dir'] / 'history.csv')
    save_json(config['result_dir'] / 'train_metrics.json', train_metrics)
    save_json(config['result_dir'] / 'val_metrics.json', val_metrics)
    save_json(config['result_dir'] / 'test_metrics.json', test_metrics)

    save_prediction_samples(
        method=method,
        test_path=DATA_PATHS['test'],
        output_dir=str(config['result_dir'] / 'predictions'),
        max_samples=5
    )

    history_df = pd.read_csv(config['result_dir'] / 'history.csv')
    return {
        'model': config['display_name'],
        'train_metrics': train_metrics,
        'val_metrics': val_metrics,
        'test_metrics': test_metrics,
        'history': history_df,
        'result_dir': config['result_dir']
    }

def print_results(title, result_df):
    print('=' * 130)
    print(title)
    print('=' * 130)
    print(f"{'Model':<30} {'IoU':<10} {'F1':<10} {'Precision':<10} {'Recall':<10} {'Accuracy':<10}")
    print('=' * 130)
    for _, row in result_df.iterrows():
        print(f"{row['model']:<30} {row['iou_plants']:<10.4f} {row['f1_plants']:<10.4f} {row['precision_plants']:<10.4f} {row['recall_plants']:<10.4f} {row['accuracy']:<10.4f}")

def plot_horizontal_comparison(result_df, title, figsize=(10, 4)):
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    metrics = ['iou_plants', 'f1_plants', 'accuracy']
    colors = ['#457b9d', '#e76f51', '#2a9d8f', '#264653']

    for ax, metric in zip(axes, metrics):
        sns.barplot(data=result_df, x='model', y=metric, ax=ax, palette=colors[:len(result_df)])
        ax.set_title(metric)
        ax.set_ylim(0, 1)
        ax.tick_params(axis='x', rotation=20)

    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

def plot_training_curves(results, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for result in results:
        history = result['history']
        axes[0].plot(history['epoch'], history['train_loss'], marker='o', label=result['model'])
        axes[1].plot(history['epoch'], history['val_iou'], marker='o', label=result['model'])

    axes[0].set_title('Train Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[1].set_title('Validation IoU')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('IoU')
    axes[0].legend()
    axes[1].legend()
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

# 4. Run the main deep learning experiments and compare the models horizontally

In [ ]:
main_results = []
for config in MAIN_MODEL_CONFIGS:
    print('\nRunning:', config['display_name'])
    main_results.append(run_single_model_experiment(config))

main_result_df = build_comparison_table(main_results)
print_results('Main Deep Learning Comparison', main_result_df)
plot_horizontal_comparison(main_result_df, 'Main Deep Learning Comparison')
plot_training_curves(main_results, 'Main Deep Learning Training Curves')

# 5. Visualize the prediction outputs of the three models

In [ ]:
for result in main_results:
    print(result['model'])
    prediction_dir = result['result_dir'] / 'predictions'
    sample_files = sorted(prediction_dir.glob('sample_*.png'))[:3]

    fig, axes = plt.subplots(1, len(sample_files), figsize=(4 * max(len(sample_files), 1), 4))
    if len(sample_files) == 1:
        axes = [axes]

    for ax, sample_file in zip(axes, sample_files):
        image = plt.imread(sample_file)
        ax.imshow(image)
        ax.set_title(sample_file.stem)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

# 6. Run the ablation studies

In [ ]:
ablation_outputs = {}

for ablation_title, ablation_configs in ABLATION_CONFIGS.items():
    print('\n' + '#' * 100)
    print(ablation_title)
    print('#' * 100)

    ablation_results = []
    for config in ablation_configs:
        print('Running:', config['display_name'])
        ablation_results.append(run_single_model_experiment(config))

    ablation_df = build_comparison_table(ablation_results)
    ablation_outputs[ablation_title] = {
        'results': ablation_results,
        'table': ablation_df
    }

    print_results(ablation_title, ablation_df)
    plot_horizontal_comparison(ablation_df, ablation_title, figsize=(9, 4))

# 7. Summarize the final results for the report

In [ ]:
summary_tables = {'Main Experiment': main_result_df}
for key, value in ablation_outputs.items():
    summary_tables[key] = value['table']

for title, table in summary_tables.items():
    print('\n' + '=' * 120)
    print(title)
    print('=' * 120)
    display(table)

## Brief Summary

U-Net achieved the best main comparison result, with an IoU of 0.7660, F1-score of 0.8599, and accuracy of 0.9433. Attention U-Net performed similarly, while DeepLabV3+ was slightly lower. The ablation studies show that pretrained encoders and data augmentation generally improve segmentation performance. In particular, data augmentation significantly improved Attention U-Net, increasing IoU from 0.7326 to 0.7750. The attention module comparison showed only a small difference between U-Net and Attention U-Net, suggesting that attention gates did not consistently improve performance in this setting. Overall, U-Net-based models were the most effective for this agricultural segmentation task.
